# 01 Smoke Test

End-to-end 검증: config 로딩 -> server 기동 -> API 호출 -> 평가 -> 결과 출력.

로직은 전부 `tq_bench` 패키지에 있고, 이 노트북은 호출만 합니다.

In [ ]:
import os, sys, logging
from pathlib import Path
from dataclasses import replace

# Ensure bench/ is importable
BENCH_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BENCH_DIR))

# LD_LIBRARY_PATH for CUDA shared libs
PROJECT_ROOT = BENCH_DIR.parent
cuda_lib = str(PROJECT_ROOT / "llama.cpp" / "build" / "bin")
os.environ["LD_LIBRARY_PATH"] = cuda_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", datefmt="%H:%M:%S")

print("bench dir:", BENCH_DIR)
print("project root:", PROJECT_ROOT)

## 1. Import and verify modules

In [ ]:
from tq_bench.config import load_runtimes, load_benchmarks, load_models, build_matrix, ExperimentCell
from tq_bench.runner import BenchmarkRunner, RunRecord
from tq_bench.server import LlamaServer, ServerLaunchConfig
from tq_bench.client import LlamaApiClient
from tq_bench.datasets import get_dataset, list_benchmarks
from tq_bench.evaluators import get_evaluator

print("All imports OK")
print(f"Available benchmarks: {list_benchmarks()}")

## 2. Load configs and show matrix size

In [ ]:
configs_dir = BENCH_DIR / "configs"

all_runtimes = load_runtimes(configs_dir / "runtimes.yaml")
all_benchmarks = load_benchmarks(configs_dir / "benchmarks.yaml")
all_models = load_models(configs_dir / "models.yaml")

matrix = build_matrix(all_runtimes, all_benchmarks)
full_cells = len(matrix) * len(all_models)

print(f"Runtimes:   {len(all_runtimes)}")
print(f"Benchmarks: {len(all_benchmarks)}")
print(f"Models:     {len(all_models)}")
print(f"Full matrix across models: {full_cells} cells")
print()
for model_id, model in all_models.items():
    print(
        f"  model={model_id:22s} gpu={str(model.gpu_id):>2s} "
        f"port={str(model.port):>4s} parallel={str(model.parallel_requests):>2s}"
    )
print()
for rt in all_runtimes:
    print(f"  {rt.id:12s}  K={rt.cache_type_k:8s}  V={rt.cache_type_v:8s}  bits={rt.bits}")

## 3. Run smoke test

기본값은 Thinking/Instruct 두 모델을 각자 `models.yaml`에 지정된 GPU/port에 맞춰 순서대로 실행합니다.

In [ ]:
# --- Smoke test parameters ---
SMOKE_MODELS = ["qwen3_vl_2b_thinking", "qwen3_vl_2b_instruct"]
SMOKE_RUNTIMES = ["baseline", "tq-4"]
SMOKE_BENCHMARK = "commonsenseqa"
N_SAMPLES = 5
OVERRIDE_GPU_ID = None   # set only when testing a single model
OVERRIDE_PORT = None     # set only when testing a single model

# --- Setup ---
rt_map = {r.id: r for r in all_runtimes}
bm_map = {b.id: b for b in all_benchmarks}
binary = PROJECT_ROOT / "llama.cpp" / "build" / "bin" / "llama-server"

if len(SMOKE_MODELS) > 1 and (OVERRIDE_GPU_ID is not None or OVERRIDE_PORT is not None):
    raise ValueError("GPU/port override only makes sense for a single selected model")

runner_cache = {}
records: list[RunRecord] = []
bm = bm_map[SMOKE_BENCHMARK]
bm_smoke = replace(bm, sample_count=N_SAMPLES)

total = len(SMOKE_MODELS) * len(SMOKE_RUNTIMES)
step = 0

for model_id in SMOKE_MODELS:
    model_config = all_models[model_id]
    gpu_id = OVERRIDE_GPU_ID if OVERRIDE_GPU_ID is not None else model_config.gpu_id
    port = OVERRIDE_PORT if OVERRIDE_PORT is not None else (model_config.port or 8080)
    parallel_requests = model_config.parallel_requests or 4

    runner_key = (port, parallel_requests)
    if runner_key not in runner_cache:
        runner_cache[runner_key] = BenchmarkRunner(
            server_binary=binary,
            default_port=port,
            request_timeout=120.0,
            max_retries=2,
            max_tokens=256,
            parallel_requests=parallel_requests,
        )
    runner = runner_cache[runner_key]

    for rid in SMOKE_RUNTIMES:
        step += 1
        rt = rt_map[rid]
        cell = ExperimentCell(runtime=rt, benchmark=bm_smoke, model_id=model_config.id)
        print(f"\n--- [{step}/{total}] {model_config.id} :: {rid} x {SMOKE_BENCHMARK} (n={N_SAMPLES}) ---")
        print(f"    gpu={gpu_id} port={port} parallel={parallel_requests}")
        record = runner.run_cell(
            cell,
            model_config,
            gpu_id=gpu_id,
            port=port,
            parallel_requests=parallel_requests,
        )
        records.append(record)
        print(f"    status={record.status}  score={record.score:.3f}  time={record.wall_time_seconds:.1f}s")

## 4. Results table

In [ ]:
import pandas as pd

rows = []
for r in records:
    rows.append({
        "model": r.model_id,
        "runtime": r.runtime_id,
        "benchmark": r.benchmark_id,
        "status": r.status,
        "score": r.score,
        "succeeded": f"{r.n_succeeded}/{r.n_samples}",
        "wall_time_s": round(r.wall_time_seconds, 1),
        "notes": r.notes,
    })

df = pd.DataFrame(rows)
display(df)

# Per-sample detail
for r in records:
    print(f"\n--- {r.model_id} :: {r.runtime_id} x {r.benchmark_id} ---")
    for sr in r.sample_results:
        mark = "OK" if sr.error is None else "ERR"
        print(f"  [{mark}] id={sr.sample_id}  score={sr.score:.2f}  pred={sr.prediction!r:.60s}  ref={sr.reference!r:.60s}")